In [1]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

In [2]:
import json
import gymnasium as gym
import matplotlib.pyplot as plt

from stable_baselines3.common.evaluation import evaluate_policy

from train.train_cartpole import train_snn_cartpole
from train.train_acrobot import train_snn_acrobot
from train.train_pendulum import train_snn_pendulum
from train.train_ann import train_ann

In [3]:
def evaluate_model(model, env_name, n_eval_episodes=10):

    env = gym.make(env_name)
    env.reset(seed=42)


    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=n_eval_episodes,
        deterministic=True
    )

    return mean_reward, std_reward


def compute_param_count(model):
    return sum(p.numel() for p in model.policy.parameters())


def compute_grad_norm(model):

    total_norm = 0

    for p in model.policy.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2

    return total_norm ** 0.5


def extract_spike_rate(model):

    extractor = model.policy.features_extractor

    if hasattr(extractor, "latest_spike_rate"):
        return extractor.latest_spike_rate

    return None

results = {}

### CartPole

In [ ]:
print("===== CartPole SNN =====")

model_snn = train_snn_cartpole()

mean_snn, std_snn = evaluate_model(model_snn, "CartPole-v1")

snn_params = compute_param_count(model_snn)
snn_grad = compute_grad_norm(model_snn)
snn_spike = extract_spike_rate(model_snn)

print("Reward:", mean_snn, "±", std_snn)
print("Parameters:", snn_params)
print("Gradient Norm:", snn_grad)
print("Spike Rate:", snn_spike)

===== CartPole SNN =====


In [ ]:
print("===== CartPole ANN =====")

model_ann = train_ann("CartPole-v1",100_000)

mean_ann, std_ann = evaluate_model(model_ann, "CartPole-v1")

ann_params = compute_param_count(model_ann)

print("Reward:", mean_ann, "±", std_ann)
print("Parameters:", ann_params)

results["CartPole-v1"] = {
    "SNN_reward_mean": mean_snn,
    "SNN_reward_std": std_snn,
    "ANN_reward_mean": mean_ann,
    "ANN_reward_std": std_ann,
    "SNN_params": snn_params,
    "ANN_params": ann_params,
    "SNN_grad_norm": snn_grad,
    "SNN_spike_rate": snn_spike
}

### Acrobot

In [ ]:
print("===== Acrobot SNN =====")

model_snn = train_snn_acrobot()

mean_snn, std_snn = evaluate_model(model_snn, "Acrobot-v1")

snn_params = compute_param_count(model_snn)
snn_grad = compute_grad_norm(model_snn)
snn_spike = extract_spike_rate(model_snn)

print("Reward:", mean_snn, "±", std_snn)
print("Parameters:", snn_params)
print("Gradient Norm:", snn_grad)
print("Spike Rate:", snn_spike)

In [ ]:
print("===== Acrobot ANN =====")

model_ann = train_ann("Acrobot-v1",200_000)

mean_ann, std_ann = evaluate_model(model_ann, "Acrobot-v1")

ann_params = compute_param_count(model_ann)

print("Reward:", mean_ann, "±", std_ann)
print("Parameters:", ann_params)

results["Acrobot-v1"] = {
    "SNN_reward_mean": mean_snn,
    "SNN_reward_std": std_snn,
    "ANN_reward_mean": mean_ann,
    "ANN_reward_std": std_ann,
    "SNN_params": snn_params,
    "ANN_params": ann_params,
    "SNN_grad_norm": snn_grad,
    "SNN_spike_rate": snn_spike
}

### Pendulum

In [ ]:
print("===== Pendulum SNN =====")

model_snn = train_snn_pendulum()

mean_snn, std_snn = evaluate_model(model_snn, "Pendulum-v1")

snn_params = compute_param_count(model_snn)
snn_grad = compute_grad_norm(model_snn)
snn_spike = extract_spike_rate(model_snn)

print("Reward:", mean_snn, "±", std_snn)
print("Parameters:", snn_params)
print("Gradient Norm:", snn_grad)
print("Spike Rate:", snn_spike)

In [ ]:
print("===== Pendulum ANN =====")

model_ann = train_ann("Pendulum-v1",200_000)

mean_ann, std_ann = evaluate_model(model_ann, "Pendulum-v1")

ann_params = compute_param_count(model_ann)

print("Reward:", mean_ann, "±", std_ann)
print("Parameters:", ann_params)

results["Pendulum-v1"] = {
    "SNN_reward_mean": mean_snn,
    "SNN_reward_std": std_snn,
    "ANN_reward_mean": mean_ann,
    "ANN_reward_std": std_ann,
    "SNN_params": snn_params,
    "ANN_params": ann_params,
    "SNN_grad_norm": snn_grad,
    "SNN_spike_rate": snn_spike
}

### Save Results

In [ ]:
with open("results/phase2_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Metrics saved")

In [ ]:
envs = list(results.keys())

snn_rewards = [results[e]["SNN_reward_mean"] for e in envs]
ann_rewards = [results[e]["ANN_reward_mean"] for e in envs]

x = range(len(envs))

plt.figure()

plt.bar(x, snn_rewards, width=0.4, label="SNN")
plt.bar([i + 0.4 for i in x], ann_rewards, width=0.4, label="ANN")

plt.xticks([i + 0.2 for i in x], envs)

plt.ylabel("Mean Reward")
plt.title("SNN vs ANN Performance")

plt.legend()

plt.savefig("results/reward_comparison.png")

plt.show()

In [ ]:
spike_rates = [results[e]["SNN_spike_rate"] for e in envs]

plt.figure()

plt.bar(envs, spike_rates)

plt.ylabel("Spike Rate")
plt.title("SNN Spike Activity")

plt.savefig("results/spike_rates.png")

plt.show()

In [ ]:
snn_params = [results[e]["SNN_params"] for e in envs]
ann_params = [results[e]["ANN_params"] for e in envs]

x = range(len(envs))

plt.figure()

plt.bar(x, snn_params, width=0.4, label="SNN")
plt.bar([i + 0.4 for i in x], ann_params, width=0.4, label="ANN")

plt.xticks([i + 0.2 for i in x], envs)

plt.ylabel("Parameter Count")
plt.title("Model Complexity")

plt.legend()

plt.savefig("results/parameter_counts.png")

plt.show()